In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [ ]:
listings_df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .load("../data/Listings.csv")
    # .load("hdfs://master-node:9000/AirbnbData/Listings.csv")  # dùng khi master-node bật
listings_df_raw.show(truncate=False)

In [ ]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import ArrayType, StringType
listings_df = listings_df_raw.withColumn(
    "amenities",
    from_json(col("amenities"), ArrayType(StringType()))
)

In [ ]:
from pyspark.sql.functions import when, col
# Quy đổi toàn bộ giá từ tiền tệ địa phương (Local Currency) sang USD
# Tỷ giá được cập nhật theo thời điểm hiện tại
listings_df_currency = listings_df.withColumn(
    "price_usd",
    when(col("city") == "New York", col("price") * 1.0)           # USD giữ nguyên
    .when(col("city") == "Paris", col("price") * 1.156)            # EUR -> USD
    .when(col("city") == "Rome", col("price") * 1.156)             # EUR -> USD
    .when(col("city") == "Sydney", col("price") * 0.7038)           # AUD -> USD
    .when(col("city") == "Hong Kong", col("price") * 0.1276)        # HKD -> USD
    .when(col("city") == "Bangkok", col("price") * 0.03048)         # THB -> USD
    .when(col("city") == "Mexico City", col("price") * 0.05795)     # MXN -> USD
    .when(col("city") == "Cape Town", col("price") * 0.06130)       # ZAR -> USD
    .when(col("city") == "Istanbul", col("price") *  0.02162)        # TRY -> USD
    .when(col("city") == "Rio de Janeiro", col("price") * 0.1961 )   # BRL -> USD
    .otherwise(col("price"))
)

In [ ]:
reviews_df=spark.read.csv("../data/Reviews.csv",header=True,inferSchema=True)
# reviews_df=spark.read.csv("hdfs://master-node:9000/AirbnbData/Reviews.csv",header=True,inferSchema=True)  # dùng khi master-node bật
reviews_df.show()

In [ ]:
listings_df_currency.createOrReplaceTempView("listings")
reviews_df.createOrReplaceTempView("reviews")
spark.catalog.listTables()

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler, Imputer
from pyspark.ml.classification import (
    LogisticRegression, DecisionTreeClassifier,
    RandomForestClassifier, LinearSVC
)
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
    ClusteringEvaluator
)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["font.family"] = "DejaVu Sans"
from sklearn.preprocessing import StandardScaler
from pyspark.sql.functions import col, regexp_replace
from pyspark.sql.types import DoubleType
import seaborn as sns
from sklearn.metrics import roc_curve, auc
from pyspark.ml.functions import vector_to_array
from sklearn.metrics import confusion_matrix
import numpy as np

print("Import thư viện thành công.")

In [ ]:
# ── Đặc trưng số (Numeric Features) ──────────────────────────────────────────
NUMERIC_COLS = [
    "price_usd",
    "host_total_listings_count",
    "minimum_nights",
    "review_scores_rating",
    "review_scores_cleanliness",
    "review_scores_communication",
]

# ── Đặc trưng phân loại (Categorical Features) ───────────────────────────────
CATEGORICAL_COLS = [
    "host_response_time",
    "instant_bookable",
    "host_identity_verified",
    "room_type",
]

# ── Cột mục tiêu ─────────────────────────────────────────────────────────────
TARGET_COL = "host_is_superhost"

print("Cấu hình hoàn tất.")
print(f"   Numeric   : {len(NUMERIC_COLS)} cột")
print(f"   Categoric : {len(CATEGORICAL_COLS)} cột")

In [ ]:
from pyspark.sql.functions import col, when, count, lit

# 1. Gán DataFrame và lọc bỏ dữ liệu thiếu
df = listings_df_currency.dropna(subset=[TARGET_COL, 'review_scores_rating'])

# 2. Tạo biến mục tiêu (Label Encoding)
df = df.withColumn("label", when(col(TARGET_COL) == "t", 1.0).otherwise(0.0))

# 3. Tính class_weight tối ưu (chỉ quét dữ liệu trên HDFS đúng 1 lần)
# Sử dụng groupBy kết hợp agg để tránh các lệnh .count() riêng lẻ
counts = df.groupBy("label").agg(count("*").alias("count")).collect()
counts_dict = {row["label"]: row["count"] for row in counts}

superhost_count = counts_dict.get(1.0, 0)
total_count = sum(counts_dict.values())
non_superhost_count = total_count - superhost_count

# Tránh lỗi chia cho 0 nếu dữ liệu thiếu class tương ứng
weight_positive = total_count / (2 * superhost_count) if superhost_count > 0 else 0
weight_negative = total_count / (2 * non_superhost_count) if non_superhost_count > 0 else 0

# Gán trọng số
df = df.withColumn(
    "class_weight",
    when(col("label") == 1.0, lit(weight_positive)).otherwise(lit(weight_negative))
)

print("Đã xử lý xong biến df! Sẵn sàng cho Phần 2.")

Tiền xử lý dữ liệu

In [ ]:
def build_preprocessing_stages(with_scaling=False):
    """
    Tạo danh sách các bước tiền xử lý.
    - with_scaling=False: Dùng cho LR, DT, RF (tích hợp StandardScaler tùy chọn)
    - with_scaling=True : Bắt buộc dùng cho SVM
    """
    stages = []

    # Bước 1: Xử lý dữ liệu khuyết thiếu (Imputer)
    imputer = Imputer(
        inputCols=NUMERIC_COLS,
        outputCols=NUMERIC_COLS
    ).setStrategy("median")
    stages.append(imputer)

    # Bước 2: Mã hóa biến phân loại (StringIndexer)
    indexed_cat_cols = []
    for col_name in CATEGORICAL_COLS:
        out_col = f"{col_name}_indexed"
        indexer = StringIndexer(
            inputCol=col_name,
            outputCol=out_col,
            handleInvalid="keep"
        )
        stages.append(indexer)
        indexed_cat_cols.append(out_col)

    # Bước 3: Đóng gói đặc trưng (VectorAssembler)
    feature_cols = NUMERIC_COLS + indexed_cat_cols
    raw_output = "raw_features" if with_scaling else "features"
    assembler = VectorAssembler(
        inputCols=feature_cols,
        outputCol=raw_output
    )
    stages.append(assembler)

    # Bước 4 (Tuỳ chọn): Chuẩn hóa dữ liệu (StandardScaler)
    # Bắt buộc với SVM vì thuật toán tính khoảng cách rất nhạy với thang đo
    if with_scaling:
        scaler = StandardScaler(
            inputCol="raw_features",
            outputCol="features",
            withStd=True,
            withMean=True
        )
        stages.append(scaler)

    return stages, feature_cols

print(" Hàm build_preprocessing_stages() đã sẵn sàng.")
print(f"   Số đặc trưng số  : {len(NUMERIC_COLS)}")
print(f"   Số đặc trưng loại: {len(CATEGORICAL_COLS)}")
print(f"   Tổng chiều vector: {len(NUMERIC_COLS) + len(CATEGORICAL_COLS)}")

Ép kiểu cho các dữ liệu số

In [ ]:
print("Đang xử lý và ép kiểu dữ liệu...")

# 2. Ép kiểu toàn bộ các cột đặc trưng số về chuẩn DoubleType
for c in NUMERIC_COLS:
    df = df.withColumn(c, col(c).cast(DoubleType()))

# 3. Kiểm tra lại schema để chắc chắn đã thành số (Double)
print("Đã ép kiểu xong! Cấu trúc các cột số hiện tại:")
df.select(NUMERIC_COLS).printSchema()

Chia tập dữ liệu

Lưu tập dữ li

Hàm tính các chỉ số đánh giá mô hình và lưu vào hàm chung để nữa show ra theo bảng tổng hợp 4 mô hình